In [65]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import InstanceHardnessThreshold
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score
from torch.nn.init import xavier_normal_
from torch.optim import Adam

In [2]:
'''CONFIG FOR LOCAL / CLOUD RUNNING'''
running_local = 'content' not in os.getcwd()
if running_local:
    path = ''
else:
    from google.colab import drive
    drive.mount('/content/drive')
    path = 'drive/MyDrive/StructuralBioinformatics/'

Mounted at /content/drive


In [68]:
df = pd.read_csv(path + 'data/contact_df.csv')

In [6]:
df.shape

(454193, 35)

In [4]:
df = df[df.Interaction.notna()]

In [8]:
df.shape

(454193, 35)

In [69]:
di = {"HBOND": 0, "IONIC": 1, "PICATION": 2, "PIPISTACK": 3, "SSBOND": 4, "VDW": 5}
df = df.replace({"Interaction": di})

In [70]:
  y = df['Interaction']
  X = df[['s_up', 's_down', 's_phi', 's_psi', 's_a1', 's_a2', 's_a3', 's_a4', 's_a5', 't_up', 't_down', 't_phi', 't_psi', 't_a1', 't_a2', 't_a3', 't_a4', 't_a5']]

In [24]:
X.isna().any()

s_up      False
s_down    False
s_phi     False
s_psi     False
s_a1      False
s_a2      False
s_a3      False
s_a4      False
s_a5      False
t_up      False
t_down    False
t_phi     False
t_psi     False
t_a1      False
t_a2      False
t_a3      False
t_a4      False
t_a5      False
dtype: bool

In [71]:
  sfm = SelectFromModel(LogisticRegression(max_iter=1000))
  sfm.fit(X, y)
  X_sub = sfm.transform(X)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
sfm.get_support()

array([False, False, False, False,  True, False, False,  True,  True,
       False, False, False, False,  True, False, False,  True,  True])

In [39]:
X_sub.shape[1]

6

In [72]:
  scaler = MinMaxScaler()
  scaler.fit(X_sub)
  X = scaler.transform(X_sub)

In [73]:
labels = set(y)
num_classes = len(labels)
features = X_sub.shape[1]
y_cat = pd.get_dummies(y)#to_categorical(y, num_classes)
y = y.values

print("Undersampling data...")
undersample = InstanceHardnessThreshold(estimator=AdaBoostClassifier(),sampling_strategy={0:50000,5:50000})
X,y = undersample.fit_resample(X, y)
print("Oversampling data...")
oversample = SMOTE(sampling_strategy={1:50000,3:50000,2:50000,4:50000})
X, y = oversample.fit_resample(X, y)


Undersampling data...
Oversampling data...


In [74]:
class NeuralNet(nn.Module):
  def __init__(self, n_layers, neurons, input_dim, output_dim):
    super(NeuralNet, self).__init__()
    self.layers = nn.ModuleList()
    for i in range(n_layers):
      if i == 0:
        self.layers.append(nn.Linear(input_dim, neurons))
      else:
        self.layers.append(nn.Linear(neurons, neurons))
      self.layers.append(nn.ReLU())

    self.layers.append(nn.Linear(neurons, output_dim))
    self.layers.append(nn.Softmax(dim=1))

  def forward(self, x):
    for layer in self.layers:
      x = layer(x)
    return x

  def train_model(self, train_loader, criterion, optimizer, epochs=30):
    for epoch in range(epochs):
      for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = self(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
      print(epoch)
      print(loss)
    return loss.item()

  def evaluate_model(self, val_loader, criterion):
    self.eval()
    with torch.no_grad():
      for inputs, labels in val_loader:
        val_outputs = self(inputs)
        val_loss = criterion(val_outputs, labels)
        _, preds = torch.max(val_outputs, 1)
        val_auc = roc_auc_score(labels.numpy(), preds.numpy(), multi_class='ovr')
    return val_loss.item(), val_auc


input_dim = features# number of input features
output_dim = num_classes# number of output classes
layers = [2, 3, 4]
neurons = [60, 90, 120]
hyperparameter_score_list = {}
histories = {}

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

train_data = TensorDataset(torch.tensor(X_train, dtype=torch.float), torch.tensor(y_train, dtype=torch.long))
val_data = TensorDataset(torch.tensor(X_val, dtype=torch.float), torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_data, batch_size=16000, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16000)

for l in layers:
    for n in neurons:
        model = NeuralNet(l, n, input_dim, output_dim)
        for module in model.modules():
            if isinstance(module, nn.Linear):
                xavier_normal_(module.weight)
        criterion = nn.CrossEntropyLoss()
        optimizer = Adam(model.parameters())

        print(f'l: {l}, n: {n}')

        # Train
        train_loss = model.train_model(train_loader, criterion, optimizer)

        # Evaluate
        val_loss, val_auc = model.evaluate_model(val_loader, criterion)

        histories[(l, n)] = train_loss
        hyperparameter_score_list[(l, n)] = val_auc

result = list(hyperparameter_score_list.items())
result.sort(key=lambda item: item[1], reverse=True)
bestresult = result[0][0]
print(f"The best parameter configuration for Neural Network is: {bestresult[0]} (hidden) layers, {bestresult[1]} neurons")


l: 2, n: 60
0
tensor(1.7605, grad_fn=<NllLossBackward0>)
1
tensor(1.7172, grad_fn=<NllLossBackward0>)
2
tensor(1.6731, grad_fn=<NllLossBackward0>)
3
tensor(1.6340, grad_fn=<NllLossBackward0>)
4
tensor(1.5692, grad_fn=<NllLossBackward0>)
5
tensor(1.4783, grad_fn=<NllLossBackward0>)
6
tensor(1.4180, grad_fn=<NllLossBackward0>)
7
tensor(1.3739, grad_fn=<NllLossBackward0>)
8
tensor(1.3424, grad_fn=<NllLossBackward0>)
9
tensor(1.3176, grad_fn=<NllLossBackward0>)
10
tensor(1.3032, grad_fn=<NllLossBackward0>)
11
tensor(1.2881, grad_fn=<NllLossBackward0>)
12
tensor(1.2806, grad_fn=<NllLossBackward0>)
13
tensor(1.2768, grad_fn=<NllLossBackward0>)
14
tensor(1.2744, grad_fn=<NllLossBackward0>)
15
tensor(1.2576, grad_fn=<NllLossBackward0>)
16
tensor(1.2544, grad_fn=<NllLossBackward0>)
17
tensor(1.2518, grad_fn=<NllLossBackward0>)


KeyboardInterrupt: ignored

In [82]:
pd.get_dummies(y).values

array([[1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0]], dtype=uint8)

In [91]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]
performance = []
y_cat = pd.get_dummies(y).values
kf = StratifiedKFold(n_splits = 10)
for train_index, test_index in kf.split(X,y):
  X_train, X_test = torch.tensor(X[train_index]), torch.tensor(X[test_index])
  y_train, y_test = torch.tensor(y_cat[train_index]), torch.tensor(y_cat[test_index])
  train_dataset = CustomDataset(X_train.float(), y_train)
  test_dataset = CustomDataset(X_test.float(), y_test)
  train_loader = DataLoader(dataset=train_dataset, batch_size=16000, shuffle=True)
  test_loader = DataLoader(dataset=test_dataset, batch_size=16000)
  #X_train, X_test = X[train_index], X[test_index]
  #y_cat_train, y_cat_test = y_cat[train_index], y_cat[test_index]
  bestmodel = NeuralNet(4, 128, input_dim, output_dim)
  criterion = nn.CrossEntropyLoss()
  optimizer = Adam(bestmodel.parameters())
  bestmodel.train_model(train_loader, criterion, optimizer, epochs=25)
  #bestmodel.fit(X_train, y_cat_train, input_dim, output_dim, epochs=25, verbose=0,batch_size=16000)
  performance.append(bestmodel.evaluate(X_test, y_cat_test)[1])

bestperformance = np.mean(performance)
print(f"The best score is equal to = {bestperformance}")

RuntimeError: ignored

In [48]:
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import functional as F
from sklearn.model_selection import train_test_split
from torch import manual_seed

def NN_Builder(n_layers, neurons, num_features, num_classes):

    manual_seed(123)

    class NeuralNetwork(nn.Module):
        def __init__(self):
            super(NeuralNetwork, self).__init__()

            self.layers = nn.ModuleList()
            self.layers.append(nn.Linear(num_features, neurons))
            for i in range(n_layers - 1):
                self.layers.append(nn.Linear(neurons, neurons))
            self.out = nn.Linear(neurons, num_classes)

        def forward(self, x):
            for layer in self.layers:
                x = F.relu(layer(x))
            return F.softmax(self.out(x), dim=1)

    return NeuralNetwork()

def train(model, criterion, optimizer, dataloader, val_dataloader, num_epochs=30, patience=5, min_delta=0.0001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_loss = None
    curr_patience = 0

    for epoch in range(num_epochs):
        model.train()

        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs = inputs.float().to(device)
            labels = labels.long().to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        # Validation loss
        model.eval()
        val_running_loss = 0.0
        for inputs, labels in val_dataloader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * inputs.size(0)

        avg_train_loss = running_loss / len(dataloader.dataset)
        avg_val_loss = val_running_loss / len(val_dataloader.dataset)

        # Early stopping
        if best_loss is None:
            best_loss = avg_val_loss
        elif best_loss - avg_val_loss > min_delta:
            best_loss = avg_val_loss
            curr_patience = 0
        elif best_loss - avg_val_loss < min_delta:
            curr_patience += 1

        if curr_patience >= patience:
            print("Early stopping due to lack of improvement in validation loss.")
            break

        print(f"Epoch {epoch+1}/{num_epochs} - Training loss: {avg_train_loss} - Validation loss: {avg_val_loss}")

layers = [2, 3, 4]
neurons = [60, 90, 120]
hyperparameter_score_list = {}

print("Determining the best model...")

for l in layers:
    for n in neurons:
        print((l, n))
        model = NN_Builder(l, n, features, num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters())

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1)
        train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
        val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

        train_dataloader = DataLoader(train_dataset, batch_size=16000)
        val_dataloader = DataLoader(val_dataset, batch_size=16000)

        train(model, criterion, optimizer, train_dataloader, val_dataloader)
        _, preds = torch.max(model(torch.from_numpy(X_val)), 1)
        acc = (preds == y_val).sum().item() / len(y_val)
        hyperparameter_score_list[(l, n)] = acc

bestresult = max(hyperparameter_score_list, key=hyperparameter_score_list.get)
print(f"The best parameter configuration for Neural Network is: {bestresult[0]} (hidden) layers, {bestresult[1]} neurons")


Determining the best model...
(2, 60)


RuntimeError: ignored